# Tail budget: instrument feasibility

## Question and why it matters
Does a transferable interior survival law explain boundary occupation?

This notebook reports instrument calibration and a prospective feasibility pilot. Scientific confirmation is not run. See [README](README.md) and [mechanism note](../../../notes/kappa-mechanisms-2026-10-15.md).

## Assumptions and definitions
Discovery includes first arrival; visit count means departures, including from the initial patch. Phase is (initial colour + departures) mod rule length. Completed cycles are floor(departures/n), without inventing prehistory for initial patch colours. The visited boundary includes hole edges; rim depth 0..4 is graph distance to that boundary. Census features weight cells; hazard tables weight departures. Exact fixture/property checks support these definitions. Return-hazard estimates remain observational.

## Provenance
All inputs below are small results intended for version control; this session makes no commit. The notebook never reads dumps or runs simulation. Seed identities are printed from the frozen manifest.

In [1]:
from pathlib import Path
import hashlib, json, csv
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Cargo.toml").is_file() and (p / "experiments").is_dir())
EXP = ROOT / "experiments/archive/2026-10-kappa-tail-budget"
inputs = ["pilot-v2.manifest.json", "pilot_summary.json", "pilot_features.csv", "calibration.json", "determinism.json", "REPORT.md"] + (["cohort_audit.json"] if EXP.name.endswith("field-age") else [])
for name in inputs:
    path = EXP / "results" / name
    assert path.is_file(), path
    print(name, hashlib.sha256(path.read_bytes()).hexdigest())
manifest = json.loads((EXP / "results/pilot-v2.manifest.json").read_text())
print("Simulation base commit:", manifest["git_commit"])
print("Registered UTC:", manifest["recorded_utc"])
print("Pilot seeds:", manifest["config"]["seeds"])
SEED = manifest["config"]["seeds"][0]


pilot-v2.manifest.json 22f5fdd60d7d6baf4137f4a79ee821f2654b37d268e94afddf0f55310f53593d
pilot_summary.json bb7026d34ac8b2ba62667a165031237b40185ecfaf82ebe9ec72115cd781520f
pilot_features.csv 4ae5ccd0924f0be5112233d2b4521b4855fde40dfc3d6bd68e85d1d2ff0d102b
calibration.json 8091188d4885a7947d2dcc748e1c478ea695d833b0bd6168dc25f6b9a9cc2245
determinism.json 9a054f799f22a328c41477623430b4d26b5edd7d6b274e9d9b9bfb63b8c70bfc
REPORT.md 71bffcf67c42ef4ee35108311e980b9ece009f611318cae78bc4faed13d9b818
Simulation base commit: b3a0f762ac2230b5df92188206a4d3a1081897be
Registered UTC: 2026-10-15T10:12:25.578010+00:00
Pilot seeds: [2026101501, 2026101502, 2026101503, 2026101504]


## Data and method
The Rust observation driver preserves the original walker and feeds the tested fixed-area accountant. Interior episodes intersecting a band are followed beyond its end; full durations and in-band overlap have separate columns. An unresolved return has no assumed finite upper bound. Birth, midpoint and endpoint censuses enumerate the visited rim. Heavy computation lives in the example binary and scripts/pilot.py. Matching and all seeds were fixed before new ant outcomes.

## Results
Frozen pilot: 44 rules x 4 starts, three area bands; cap 2^30, grid 4096, four workers. Base commit `b3a0f762ac2230b5df92188206a4d3a1081897be`; dirty source hashes and seed identities are in [manifest](results/pilot-v2.manifest.json).

| Band start | Completed / requested | Unresolved interior time | Minimum completed entry episodes |
|---|---:|---:|---:|
| 32768 | 176 / 176 | 0 | 1018 |
| 131072 | 176 / 176 | 0 | 4085 |
| 524288 | 176 / 176 | 0 | 16362 |

No-escape gate: **True**. All-complete gate: **True**. Quarter-cap reserve gate: **False** (max final step 831,505,321). Ant batch wall time 655.4 s; measured whole-process peak RSS 530.4 MiB (four concurrent jobs).

## Instrument validation

Four reference/property tests check discovery times, exact departure counts, nonzero patch phase offsets, visited-boundary rim census, offline fragment accounting, follow-up past band edges, right censoring, and rotor/random-walk paths. Full release tests passed; see the tail-budget results/instrument-release-tests.log.

Sampled thread replay: **True**, 8 specs and 48 files, one versus six workers at the two profile bands. Finished-trajectory resume preserves bytes. Full all-seed replay and mid-trajectory state checkpoints are **not** validated.

Known-answer renewal calibration uses 16 independent seeds per law, 20,000 cycles per seed. Mean interior durations (mean +/- between-seed sd; known mean):

- geometric: 4.011272 +/- 0.018268; known 4.000000; bias +0.011272.
- mixture: 11.448353 +/- 0.282892; known 11.500000; bias -0.051647.
- serial_mixture: 11.519500 +/- 0.447971; known 11.500000; bias +0.019500.

All three predeclared four-standard-error Monte Carlo checks pass. These calibrate the actual episode accountant, including a serially correlated mixture, not a fitted survival family.

## Split-seed feature reliability (descriptive)

Spearman correlation across rules between first two and last two pilot seeds, restricted to rules with all four starts attaining the band and the feature observed. Contributing rule counts are shown in parentheses. No bootstrap inference is made; coverage denominators retain every requested start. The CSV includes every seed and between-seed dispersion is in the summary JSON.

| Feature | Area 32768 | Area 131072 | Area 524288 |
|---|---:|---:|---:|
| entry_mean | 0.656 (44) | 0.635 (44) | 0.847 (44) |
| tail_probability_64 | 0.957 (44) | 0.962 (44) | 0.985 (44) |
| tail_mean_64 | 0.863 (42) | 0.828 (43) | 0.921 (43) |

## Meaning and remaining work

These results establish observation/accounting feasibility at the measured scales and describe repeatability. They do not test T1/T2 or A1/A2/A3, compare tail or age features with kappa, establish causal mechanisms, or calibrate a survival-family fit or drift forecast.

Confirmation remains unregistered and unlaunched. Before confirmation: implement and byte-test mid-trajectory state checkpoints; calibrate/freeze the survival family and predictor loss; freeze grouped rule folds and seed roles; resolve the failed age cohort support gate explicitly; choose seed counts using pilot feature reliability. Retain the original thresholds and report deviations prospectively. No successful scientific prediction is claimed.

## Reproduce

Use WSL from the repo root with `export UV_PROJECT_ENVIRONMENT=$HOME/.venvs/turmite UV_LINK_MODE=copy`. Source-hash guards reject changes to registered inputs; retain old manifests and use a new run label for changed sources.

```bash
cargo build --release --example kappa_tail_budget_sampler --example kappa_field_age_sampler --example kappa_tail_budget_calibrate
uv run experiments/archive/2026-10-kappa-tail-budget/scripts/calibrate.py
uv run experiments/archive/2026-10-kappa-tail-budget/scripts/run.py --name pilot-v2
uv run experiments/archive/2026-10-kappa-tail-budget/scripts/summarize.py
uv run experiments/archive/2026-10-kappa-tail-budget/scripts/verify.py
uv run experiments/archive/2026-10-kappa-tail-budget/scripts/report.py
cargo test --release
```

The pilot itself is frozen prospectively in pilot-v2.manifest.json. Its exact preparation command is recorded there. pilot-v1 was rejected before simulation because an unrelated module export changed src/lib.rs; its manifest and failure log are retained.


In [2]:
summary = json.loads((EXP / "results/pilot_summary.json").read_text())
print("Run gates:", json.dumps(summary["run_gates"], indent=2))
for b in summary["bands"]:
    print("Area", b["lo"], "completed", b["completed"], "/", b["requested"],
          "unresolved in-band I steps", b["unresolved_overlap"])
print("Scientific verdict:", summary["scientific_verdict"])


Run gates: {
  "all_complete": true,
  "elapsed_seconds": 655.442665442999,
  "max_t": 831505321,
  "no_escape": true,
  "quarter_cap_reserve": false
}
Area 32768 completed 176 / 176 unresolved in-band I steps 0
Area 131072 completed 176 / 176 unresolved in-band I steps 0
Area 524288 completed 176 / 176 unresolved in-band I steps 0
Scientific verdict: NOT TESTED: no fitted/calibrated survival family, outcome seed roles, or frozen confirmatory regression; no causal conclusion


The tables show instrument coverage and repeatability at these scales. They do not show that either mechanism predicts independent-seed kappa.

## Caveats and controls
Rotor and random walk are path/accounting controls at smaller areas, not a universal kappa calibration. Synthetic hazard labels are exogenous; label permutation is not a physical field intervention. Full state checkpoints and a frozen scientific estimator remain missing. The original age cohort matching gate failed.

## What this changes / next steps
Observation-only pilots are now feasible with measured resource cost. Freeze the estimator and resolve the cohort and checkpoint requirements before confirmation.

## Open questions
Does tail probability or conditional tail mean transfer independently? Do cell-weighted age features add predictive value beyond elapsed time, geometry, and prefix on held-out rules?

## Gate resolution (2026-10-16)
The pilot left two blockers: the quarter-cap reserve gate and the missing survival-family and checkpoint calibration. Criteria were written in NOTEBOOK.md before the measurements. Nothing below uses B, I or B/S, and T1/T2 are NOT evaluated.

In [3]:
rp = json.loads((EXP / "results/reserve_projection.json").read_text())
print("pilot max final step", rp["pilot_max_final_step"])
for k, v in rp["candidates"].items():
    print(k, "reserve", int(v["reserve"]), "pilot ok:", v["pilot_ok"], "headroom over pilot max: %.2f" % v["headroom_over_pilot_max"])
print("parametric projections DISCARDED:", rp["projection_discard_reason"])
v = rp["validation"]
print("reserve-v1:", v["completed"], "/", v["requested"], "attained, escapes", v["escaped"], "max step", v["max_t"], "within cap/4 for 2^32 and 2^33:", v["all_within_quarter_cap"])
ir = json.loads((EXP / "results/interrupt_resume.json").read_text())
print("1 vs 2 threads identical:", ir["one_vs_two_threads_identical"], "| killed at", ir["done_markers_at_kill"], "markers then resumed identical:", ir["uninterrupted_vs_resumed_identical"], "| files", ir["files_compared"])

pilot max final step 831505321
2^30 reserve 268435456 pilot ok: False headroom over pilot max: 0.32
2^32 reserve 1073741824 pilot ok: True headroom over pilot max: 1.29
2^33 reserve 2147483648 pilot ok: True headroom over pilot max: 2.58
parametric projections DISCARDED: log-normal extrapolation is invalid: per-rule sd of log t_final spans 0.006-1.6 with a 29x range in one cheap rule; extreme-value quantiles are driven by the inverse-chi2 / pooled spread of cheap rules, not by physics. Gate 1(b) is replaced by a direct fresh validation batch (see NOTEBOOK).
reserve-v1: 96 / 96 attained, escapes 0 max step 796445346 within cap/4 for 2^32 and 2^33: {'4294967296': True, '8589934592': True}
1 vs 2 threads identical: True | killed at 8 markers then resumed identical: True | files 182


Reserve gate: cap 2^32 passes (every validation start attained; headroom 1.29x over the pilot max, 1.35x over the validation max); 2^33 is a free extra margin. The registered log-normal projection was invalid (per-rule spread in final step varies 0.006-1.6 in log sd) and is kept only as a discarded diagnostic. Resume: finished-trajectory resume is byte-identical across 1/2 threads and after a SIGKILL. No mid-trajectory state checkpoint exists (needs edits to src/); the loss per interruption is bounded by the slowest trajectory. The kill landed between trajectories, so the half-written-stem path is argued, not tested.

In [4]:
fc = json.loads((EXP / "results/fit_calibration.json").read_text())
for k, v in fc["criteria"].items(): print(k, v)
print(json.dumps(fc["negative_controls"], indent=1))
t1 = json.loads((EXP / "results/t1_machinery.json").read_text())
print({k: t1[k] for k in t1 if k.startswith(("null", "power", "G4"))})

ALL_PASS False
C1_frac_cells_abs_relbias_le_0.15 0.9924242424242424
C1_median_abs_relbias_e1 0.03081281313971418
C1_median_alpha_bias -0.0029870085258020956
C1_pass True
C2_frac_cells_e1_beats_e0_rmse 0.4393939393939394
C2_pass False
C3_cells_ge30_tail_events 126
C3_misspecified_e1_vs_e0_rmse_win_frac 0.12121212121212122
C3_misspecified_median_abs_relbias_e0 0.0034893264456251905
C3_misspecified_median_abs_relbias_e1 0.14498968198736856
C3_pass False
C3_power_misspecified_real_shape_rej01_pooled 0.9720899470899471
C3_size_in_family_rej01 0.1351010101010101
C3_size_in_family_rej05 0.22058080808080807
C4_geometric20_gof_reject 0.11666666666666667
C4_lognormal_gof_reject 1.0
C4_pass False
C5_frac_cells_cov_ge_0.99 0.9318181818181818
C5_median_naive_completed_only_relbias -0.5381942015260696
C5_median_width_over_E 2.9272278260372824
C5_min_coverage 0.9
C5_pass False
pilot_split_check {'family_better_frac': 0.3650190114068441, 'median_abs_log_err_empirical': 0.21661782093283918, 'median_abs

Survival family F (truncated power law above 64 steps) is **not frozen**: it is unbiased in-family (C1 passes) but loses to the plain empirical binned mean on realistic shapes (C2 fails: 44% of cells; on the real pilot shapes it is biased by 14.5% vs 0.3%), its lack-of-fit test is over-sized (13.5% at 1%), it cannot tell an exponential cut-off from a power law (C4 fails for geometric), and censoring brackets are uninformative (C5 fails). The registered fallback applies: E[T] is the empirical binned mean; F is a sensitivity. The T1 decision machinery (synthetic) has 0 false positives and saturated power, a weak control because between-rule E[T] spread is large and the nuisance regressors are synthetic assumptions.

## confirm-v1 result and decomposition

T1 FAILS (band 0 lower bound -0.018), T2 passes (0.123) but persistence (0.072) is better. Decomposition below is a model counterfactual, largely forced by the identity E_T >> m_B; it does not support a causal tail mechanism.

In [5]:
a = json.loads((EXP / "results/confirm-v1.analysis.json").read_text())
print(json.dumps({k: a[k] for k in ("T1", "T1_pass", "T2", "T2_pass")}, indent=1))
d = json.loads((EXP / "results/confirm-v1.decomposition.json").read_text())
for o, v in d.items(): print(o, v["median_share"], v["corr_model_obs_delta"], v["only_boundary_transfers"])

{
 "T1": {
  "0": {
   "gain": 0.3523998723445718,
   "lo95": -0.01846013844920264,
   "hi95": 0.7765128953394033,
   "r2_base": -0.08495028331668508
  },
  "1": {
   "gain": 0.5106222474079932,
   "lo95": 0.25338261817998137,
   "hi95": 0.7598537609903108,
   "r2_base": 0.051139650446782814
  }
 },
 "T1_pass": false,
 "T2": {
  "identity": {
   "median_rel_err": 0.1227488831866258,
   "iqr": [
    0.05940615682301906,
    0.2546588672156574
   ]
  },
  "q999": {
   "median_rel_err": 0.5038798278318594,
   "iqr": [
    0.08805073492238039,
    0.5897206764984555
   ]
  },
  "persistence": {
   "median_rel_err": 0.07165469169892692,
   "iqr": [
    0.028481438549762934,
    0.2235389762537421
   ]
  }
 },
 "T2_pass": true
}
64 {'p_tail': 0.0723484798875607, 'tail_mean': 0.9240085805469506, 'short_mean': 0.0003312678204515369, 'boundary_mean': 0.0030612430159066566} 0.6630128493719528 False
16 {'p_tail': 0.041970916789682435, 'tail_mean': 0.9536239220377447, 'short_mean': 0.0001128296174

Caveats: analysis and decomposition scripts were written after the run (choices fixed in headers); single 44-rule cohort; no census-wide claim; short mean is a fourth factor beyond the README.